In [1]:
# =====================================================
# Final CatBoost Model - Student Health Risk Prediction
# =====================================================
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.feature_engineering import create_features
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

from src.feature_engineering import create_features

In [2]:
# ============================
# Load Training and Test Data
# ============================

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

train.head()

Train Shape: (690088, 15)
Test Shape : (295753, 14)


,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [3]:
# =====================================
# Feature Engineering & Data Preparation
# =====================================

TARGET = "health_condition"
ID_COL = "id"

# Apply feature engineering
train = create_features(train)
test = create_features(test)

# Split features and target
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]

X_test = test.drop(columns=[ID_COL])

print("Training Features:", X.shape)
print("Test Features    :", X_test.shape)
print("Target Shape     :", y.shape)

print("\nFirst 10 Features:")
print(X.columns.tolist()[:10])

Training Features: (690088, 18)
Test Features    : (295753, 18)
Target Shape     : (690088,)

First 10 Features:
['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality']


In [4]:
X_final = X.copy()
X_test_final = X_test.copy()

# categorical
for col in categorical_features:
    X_final[col] = (
        X_final[col]
        .fillna("Missing")
        .astype(str)
    )

    X_test_final[col] = (
        X_test_final[col]
        .fillna("Missing")
        .astype(str)
    )

# numerical
for col in numerical_features:

    median = X_final[col].median()

    X_final[col] = X_final[col].fillna(median)

    X_test_final[col] = X_test_final[col].fillna(median)

print("Done")

NameError: name 'categorical_features' is not defined

In [ ]:
# ==========================================
# Handle Missing Values & Categorical Columns
# ==========================================

# Detect categorical columns
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

# Fill missing values
for col in X.columns:
    if col in cat_features:
        X[col] = X[col].fillna("Unknown").astype(str)
        X_test[col] = X_test[col].fillna("Unknown").astype(str)
    else:
        median = X[col].median()
        X[col] = X[col].fillna(median)
        X_test[col] = X_test[col].fillna(median)

# Get CatBoost categorical feature indices
cat_feature_indices = [
    X.columns.get_loc(col)
    for col in cat_features
]

print(f"Total Features       : {X.shape[1]}")
print(f"Categorical Features : {len(cat_features)}")
print(f"Numerical Features   : {X.shape[1] - len(cat_features)}")

print("\nCategorical Columns:")
print(cat_features)

Total Features       : 18
Categorical Features : 7
Numerical Features   : 11

Categorical Columns:
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category']


In [6]:
# =====================================
# Load Best Hyperparameters
# =====================================

best_params = pd.read_csv("best_catboost_params.csv").iloc[0].to_dict()

# Convert data types
best_params["iterations"] = int(best_params["iterations"])
best_params["depth"] = int(best_params["depth"])
best_params["border_count"] = int(best_params["border_count"])

best_params["learning_rate"] = float(best_params["learning_rate"])
best_params["l2_leaf_reg"] = float(best_params["l2_leaf_reg"])
best_params["random_strength"] = float(best_params["random_strength"])
best_params["bagging_temperature"] = float(best_params["bagging_temperature"])

print("Best Parameters")
print(best_params)

Best Parameters
{'iterations': 3501, 'learning_rate': 0.0729819760117804, 'depth': 8, 'l2_leaf_reg': 2.6130877247254016, 'random_strength': 1.9111220326055869, 'bagging_temperature': 0.5503784078725401, 'border_count': 244}


In [8]:
# ==========================================
# CatBoost Categorical Feature Indices
# ==========================================

# Detect categorical columns
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

# Convert column names to column indices
cat_feature_indices = [
    X.columns.get_loc(col)
    for col in cat_features
]

print("Categorical Features:")
print(cat_features)

print("\nIndices:")
print(cat_feature_indices)

Categorical Features:
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category']

Indices:
[7, 8, 9, 10, 11, 12, 17]


In [9]:
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights
classes = np.unique(y)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weights = dict(zip(classes, weights))

print("Class Weights:", class_weights)

# Create model
model = CatBoostClassifier(
    iterations=best_params["iterations"],
    learning_rate=best_params["learning_rate"],
    depth=best_params["depth"],
    l2_leaf_reg=best_params["l2_leaf_reg"],
    random_strength=best_params["random_strength"],
    bagging_temperature=best_params["bagging_temperature"],
    border_count=best_params["border_count"],

    loss_function="MultiClass",
    eval_metric="TotalF1",
    class_weights=class_weights,

    random_seed=42,
    verbose=200,

    task_type="GPU"      # Change to "CPU" if GPU is unavailable
)

# Train model
model.fit(
    X,
    y,
    cat_features=cat_feature_indices
)

Class Weights: {'at-risk': np.float64(0.38819519565636845), 'fit': np.float64(5.779195873007898), 'unhealthy': np.float64(3.9849860254544613)}
0:	learn: 0.8041343	total: 64ms	remaining: 3m 43s
200:	learn: 0.9504855	total: 7.11s	remaining: 1m 56s
400:	learn: 0.9516862	total: 13.7s	remaining: 1m 45s
600:	learn: 0.9528839	total: 20.7s	remaining: 1m 39s
800:	learn: 0.9543445	total: 27.8s	remaining: 1m 33s
1000:	learn: 0.9559972	total: 34.8s	remaining: 1m 26s
1200:	learn: 0.9578026	total: 41.8s	remaining: 1m 20s
1400:	learn: 0.9598378	total: 49s	remaining: 1m 13s
1600:	learn: 0.9619509	total: 56.3s	remaining: 1m 6s
1800:	learn: 0.9639990	total: 1m 5s	remaining: 1m 1s
2000:	learn: 0.9662060	total: 1m 14s	remaining: 55.8s
2200:	learn: 0.9684404	total: 1m 22s	remaining: 48.8s
2400:	learn: 0.9703149	total: 1m 30s	remaining: 41.3s
2600:	learn: 0.9719770	total: 1m 37s	remaining: 33.7s
2800:	learn: 0.9737276	total: 1m 44s	remaining: 26.1s
3000:	learn: 0.9751119	total: 1m 51s	remaining: 18.6s
3200:

CatBoostClassifier(bagging_temperature=0.5503784078725401, border_count=244, class_weights={'at-risk': np.float64(0.38819519565636845), 'fit': np.float64(5.779195873007898), 'unhealthy': np.float64(3.9849860254544613)}, depth=8, eval_metric='TotalF1', iterations=3501, l2_leaf_reg=2.6130877247254016, learning_rate=0.0729819760117804, loss_function='MultiClass', random_seed=42, random_strength=1.9111220326055869, task_type='GPU', verbose=200)

In [10]:
# =====================================
# Predict Test Set
# =====================================

test_predictions = model.predict(X_test)

# Convert predictions to 1D array
test_predictions = test_predictions.flatten()

print("Predictions:", len(test_predictions))
print(test_predictions[:10])

Predictions: 295753
['unhealthy' 'unhealthy' 'at-risk' 'at-risk' 'unhealthy' 'fit' 'at-risk'
 'at-risk' 'at-risk' 'at-risk']


In [11]:
# =====================================
# Create Submission File
# =====================================

submission = pd.DataFrame({
    "id": test["id"],
    "health_condition": test_predictions
})

submission.to_csv("../output/submissions/submission_final_CatBoost_Gradient_Boosting.csv", index=False)

submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [12]:
model.save_model("catboost_optuna_3501iter_v1.cbm")